### Load Dataset

In [ ]:
import pandas as pd 

df = pd.read_csv('../fixed_final_data_product.csv')

### Data Cleaning

In [2]:
def clean_dataset(df):

    cols_to_drop = set()

    # 1) Substring-based drops (distance markers etc.)
    substr_drop = [
        'dist_360',
        'dist_430',
        'dist_530',
        'left_dist',
        'right_dist',
        'proj_from_ref',
    ]

    # 2) Prefix-based drops (many samples: YAW:12, PITCH:12, etc.)
    prefix_drop = [
        'CURRENTLAPTIMEINMS',
        'LAPDISTANCE',
        'ext_LAPDISTANCE',
        'ext_TIMETOINMS',
        'YAW',
        'ROLL',
        'PITCH',
        'WORLDPOSITIONX',
        'WORLDPOSITIONY',
        'WORLDFORWARDDIRX',
        'WORLDFORWARDDIRY',
    ]

    # 3) Explicit single-column drops
    explicit_drop = [
        'lap_id',
        'invalid_lap',
    ]

    for col in df.columns:
        # Always keep the target, no matter what rules say
        if col == 'Target_CURRENTLAPTIMEINMS':
            continue

        lc = col.lower()

        # --- explicit single-column drops ---
        if col in explicit_drop:
            cols_to_drop.add(col)
            continue

        # --- substring drops (distance markers, proj_from_ref, left/right_dist etc.) ---
        if any(sub in col for sub in substr_drop):
            cols_to_drop.add(col)
            continue

        # --- prefix-based drops (e.g. YAW, PITCH, etc.) ---
        for pref in prefix_drop:
            if col == pref or col.startswith(pref + "_"):
                cols_to_drop.add(col)
                break  # no need to check other prefixes

        # --- apex distance/angle drops ---
        # drop apex* columns that are distances or angles
        if 'apex' in lc:
            if any(token in lc for token in ['dist', 'distance', 'angle']):
                # e.g. apex1_distance, apex2_angle_from_ref etc.
                cols_to_drop.add(col)

    df_cleaned = df.drop(columns=[c for c in cols_to_drop if c in df.columns],
                         errors='ignore')

    return df_cleaned

df = clean_dataset(df)

target_col = "Target_CURRENTLAPTIMEINMS"
prefixes = ['BPS', 'STS', 'BPE', 'STM', 'STE', 'THE', 'THS']

def apply_iqr_filter(data, target_col, prefixes):
    """Apply IQR filtering to target and selected prefixes."""
    data = data.copy()
    # Filter target
    Q1 = data[target_col].quantile(0.25)
    Q3 = data[target_col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    data = data[data[target_col] <= upper_bound]

    # Filter chosen prefixes
    for p in prefixes:
        col = f"{p}_CURRENTLAPTIMEINMS"
        if col in data.columns:
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            data = data[data[col] <= upper_bound]
    return data

# Removing laps whose times are statistically too high
df = apply_iqr_filter(df, target_col, prefixes)

# Removing rows will Null values in any column
df = df.dropna()

In [ ]:
def clean_dataset(df):
    
    target_col = "Target_CURRENTLAPTIMEINMS"
    prefixes = ['BPS', 'STS', 'BPE', 'STM', 'STE', 'THE', 'THS']

    driver_input = ['BPS_SPEED', 'BPS_THROTTLE', 'BPS_STEER', 'BPS_BRAKE', 'BPS_CURRENTLAPTIMEINMS', 
             'BPS_LAPDISTANCE', 'BPS_WORLDPOSITIONX', 'BPS_WORLDPOSITIONY', 'BPE_SPEED', 'BPE_THROTTLE', 
             'BPE_STEER', 'BPE_BRAKE', 'BPE_CURRENTLAPTIMEINMS', 'BPE_LAPDISTANCE', 'BPE_WORLDPOSITIONX', 'BPE_WORLDPOSITIONY',
            'THS_SPEED', 'THS_THROTTLE', 'THS_STEER', 'THS_BRAKE', 'THS_CURRENTLAPTIMEINMS', 'THS_LAPDISTANCE', 'THS_WORLDPOSITIONX', 
             'THS_WORLDPOSITIONY', 'THE_SPEED', 'THE_THROTTLE', 'THE_STEER', 'THE_BRAKE', 'THE_CURRENTLAPTIMEINMS', 
             'THE_LAPDISTANCE', 'THE_WORLDPOSITIONX', 'THE_WORLDPOSITIONY', 'STS_SPEED', 'STS_THROTTLE', 'STS_STEER', 'STS_BRAKE', 
             'STS_CURRENTLAPTIMEINMS', 'STS_LAPDISTANCE', 'STS_WORLDPOSITIONX', 'STS_WORLDPOSITIONY', 'STM_SPEED', 'STM_THROTTLE', 
             'STM_STEER', 'STM_BRAKE', 'STM_CURRENTLAPTIMEINMS', 'STM_LAPDISTANCE', 'STM_WORLDPOSITIONX', 'STM_WORLDPOSITIONY', 
            'STE_SPEED', 'STE_THROTTLE', 'STE_STEER', 'STE_BRAKE', 'STE_CURRENTLAPTIMEINMS', 'STE_LAPDISTANCE', 'STE_WORLDPOSITIONX', 
             'STE_WORLDPOSITIONY', 'Target_CURRENTLAPTIMEINMS']

    # Keep only driver related features
    df = df[[col for col in driver_input if col in df.columns]] 

    return df

df = clean_dataset(df)

target_col = "Target_CURRENTLAPTIMEINMS"
prefixes = ['BPS', 'STS', 'BPE', 'STM', 'STE', 'THE', 'THS']

def apply_iqr_filter(data, target_col, prefixes):
    """Apply IQR filtering to target and selected prefixes."""
    data = data.copy()
    # Filter target
    Q1 = data[target_col].quantile(0.25)
    Q3 = data[target_col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    data = data[data[target_col] <= upper_bound]

    # Filter chosen prefixes
    for p in prefixes:
        col = f"{p}_CURRENTLAPTIMEINMS"
        if col in data.columns:
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            data = data[data[col] <= upper_bound]
    return data

# Removing laps whose times are statistically too high
df = apply_iqr_filter(df, target_col, prefixes)

# Removing rows will Null values in any column
df = df.dropna()

### Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# drop leak columns and target from predictors
X = df.drop(columns=['Target_CURRENTLAPTIMEINMS'])
X = X.drop(columns=[c for c in X.columns if 'CURRENTLAPTIMEINMS' in c])

# set target variable
y = df['Target_CURRENTLAPTIMEINMS']

# split train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train model with best hyperparameters
model = LGBMRegressor(
    random_state=42,
    verbosity=-1,
)

model.fit(X_train, y_train)

# make predictions
preds = model.predict(X_test)

# evaluate performance
n = X_test.shape[0]
p = X_test.shape[1]
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
mae  = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:,.4f}")
print(f"Adj R²: {r2_adj:,.4f}")
print(f"MAE  : {mae:,.2f}")


MSE : 42,662.96
RMSE: 206.55
R²  : 0.8935
Adj R²: 2.0551
MAE  : 132.66


### Randomised Search CV

In [10]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# train-test split
target_col = "Target_CURRENTLAPTIMEINMS"

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# base model
base_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1
)

# hyperparameter tune distributions
param_dist = {
    "num_leaves":       [31, 63, 127],
    "max_depth":        [-1, 6, 8, 10],
    "learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "n_estimators":     [500, 1000, 1500, 2000],
    "subsample":        [0.7, 0.8, 0.9, 1.0],    # bagging_fraction
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],    # feature_fraction
    "min_child_samples":[20, 40, 80, 120],
    "reg_alpha":        [0.0, 0.1, 1.0],         # L1
    "reg_lambda":       [0.0, 0.1, 1.0],         # L2
}

# randomised search CV
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=40,                    # you can reduce if too slow (e.g. 20)
    scoring="neg_mean_squared_error",
    cv=3,                         # 3-fold CV
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nBest params from RandomizedSearchCV:")
print(random_search.best_params_)

# evaluate performance
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

n = X_test.shape[0]
p = X_test.shape[1]
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, preds)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:,.4f}")
print(f"Adj R²: {r2_adj:,.4f}")
print(f"MAE  : {mae:,.2f}")


Fitting 3 folds for each of 40 candidates, totalling 120 fits

Best params from RandomizedSearchCV:
{'subsample': 0.7, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'num_leaves': 63, 'n_estimators': 1000, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
MSE : 45,229.81
RMSE: 212.67
R²  : 0.8871
Adj R²: 1.6152
MAE  : 125.67


### Grid Search

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# split train and test sets
target_col = "Target_CURRENTLAPTIMEINMS"

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# base model
base_model = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1
)

# hyperparameter tune grid
param_grid = {
    "num_leaves":       [31, 63],
    "max_depth":        [-1, 8],
    "learning_rate":    [0.01, 0.05],
    "n_estimators":     [500, 1000],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_samples":[20, 80],
    "reg_alpha":        [0.0, 0.1],
    "reg_lambda":       [0.0, 0.1],
}

# grid search CV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nBest params from GridSearchCV:")
print(grid_search.best_params_)

#  evaluate performance
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

n = X_test.shape[0]
p = X_test.shape[1]

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:,.4f}")
print(f"MAE : {mae:,.2f}")


Fitting 3 folds for each of 512 candidates, totalling 1536 fits

Best params from GridSearchCV:
{'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 500, 'num_leaves': 31, 'reg_alpha': 0.1, 'reg_lambda': 0.0, 'subsample': 0.8}
Adj R²: not defined (p >= n).
MSE : 36,363.06
RMSE: 190.69
R²  : 0.9092
MAE : 124.92


### Edited Model 

In [12]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# drop leak columns and target from predictors
X = df.drop(columns=['Target_CURRENTLAPTIMEINMS'])
X = X.drop(columns=[c for c in X.columns if 'CURRENTLAPTIMEINMS' in c])

# set target variable
y = df['Target_CURRENTLAPTIMEINMS']

# split train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train model with best hyperparameters
model = LGBMRegressor(
    random_state=42,
    verbosity=-1,
    colsample_bytree=1.0,
    learning_rate=0.01,
    max_depth=8,
    min_child_samples=20,
    n_estimators=500,
    num_leaves=31,
    reg_alpha=0.1,
    reg_lambda=0.0,
    subsample=0.8
)

model.fit(X_train, y_train)

# make predictions
preds = model.predict(X_test)

# evaluate performance
n = X_test.shape[0]
p = X_test.shape[1]
mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
mae  = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:,.4f}")
print(f"Adj R²: {r2_adj:,.4f}")
print(f"MAE  : {mae:,.2f}")

MSE : 43,525.75
RMSE: 208.63
R²  : 0.8914
Adj R²: 2.0764
MAE  : 136.43
